In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col,count, sum, date_format


In [5]:
spark = SparkSession.builder.appName("Case-study-app").getOrCreate()

In [3]:
# df1 = pd.read_csv('data/customers.csv')

In [4]:
# df1.head()

In [5]:
# print('Total records:',df1.count())

In [6]:
# df2= pd.read_csv('data/products.csv')
# df3= pd.read_csv('data/order_items.csv')
# df4= pd.read_csv('data/orders.csv')
# df5= pd.read_csv('data/returns.csv')

In [7]:
# print('Total products:', df2.count())

In [8]:
# print('Total orders:', df4.count())

In [9]:
# print('Total returns:', df5.count())

In [10]:
# print('Total order_items:', df3.count())

In [19]:
products= spark.read.option("header",True).csv("data/products.csv")
customers= spark.read.option("header",True).csv("data/customers.csv")
orders= spark.read.option("header",True).csv("data/orders.csv")
returns= spark.read.option("header",True).csv("data/returns.csv")
order_items=spark.read.option("header",True).csv("data/order_items.csv")

In [12]:
products.show()

+----------+------------+--------------+-------+---------+
|product_id|product_name|      category|  brand|unit_cost|
+----------+------------+--------------+-------+---------+
|         1|   Product_1|Home & Kitchen|Brand_A|   509.39|
|         2|   Product_2|   Electronics|Brand_A|   332.22|
|         3|   Product_3|        Sports|Brand_D|    68.58|
|         4|   Product_4|      Clothing|Brand_A|   729.19|
|         5|   Product_5|Home & Kitchen|Brand_D|   326.77|
|         6|   Product_6|        Beauty|Brand_E|   684.21|
|         7|   Product_7|          Toys|Brand_D|   634.58|
|         8|   Product_8|Home & Kitchen|Brand_B|   158.47|
|         9|   Product_9|   Electronics|Brand_C|    287.9|
|        10|  Product_10|      Clothing|Brand_A|    93.38|
|        11|  Product_11|         Books|Brand_C|   477.32|
|        12|  Product_12|        Sports|Brand_C|   517.62|
|        13|  Product_13|         Books|Brand_A|   723.64|
|        14|  Product_14|          Toys|Brand_B|   179.4

In [7]:
products.createOrReplaceTempView("Products_table")

In [14]:
result= spark.sql('''
SELECT category, SUM(unit_cost) AS Total_sales
FROM Products_table
GROUP BY category
''')

In [15]:
result.show()

[Stage 6:>                                                          (0 + 1) / 1]

+--------------+------------------+
|      category|       Total_sales|
+--------------+------------------+
|Home & Kitchen| 2901364.330000004|
|        Sports| 2853163.040000003|
|   Electronics|2864604.7399999946|
|      Clothing| 2841424.610000002|
|         Books|2853871.8500000075|
|        Beauty|2919388.7500000037|
|          Toys|2851913.1100000013|
+--------------+------------------+



In [16]:
result.write.mode("overwrite").csv("output/q2",header= True)

26/06/16 03:38:28 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
                                                                                

In [20]:
customers.createOrReplaceTempView("customers")
orders.createOrReplaceTempView("orders")
order_items.createOrReplaceTempView("order_items")
returns.createOrReplaceTempView("returns")

In [18]:
res=spark.sql('''
SELECT c.customer_name, SUM(oi.selling_price*oi.quantity) AS Total_amount
FROM order_items oi join orders o
ON oi.order_id = o.order_id
JOIN customers c
ON o.customer_id = c.customer_id
GROUP BY c.customer_name
ORDER BY Total_amount DESC
LIMIT 10
''')

In [19]:
res.show()

26/06/16 03:38:42 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 03:38:42 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


+--------------+------------------+
| customer_name|      Total_amount|
+--------------+------------------+
|Customer_93094|181569.68000000005|
|Customer_64560|169060.39999999997|
|Customer_23289|          161573.8|
|Customer_52275|153364.78999999998|
|Customer_61218|         153067.55|
|Customer_52034|         152680.05|
|Customer_40442|151037.32000000004|
|Customer_60528|         148691.95|
|Customer_84830|         148363.84|
|Customer_82593|         148281.04|
+--------------+------------------+



In [20]:
res.write.mode("overwrite").csv("output/q3",header= True)

26/06/16 03:38:54 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 03:38:54 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
                                                                                

In [31]:
#q4
res4=orders.join(order_items, on="order_id").withColumn("order_month", date_format(col("order_date"),"yyyy-MM")).groupBy("order_month").agg(sum(col("quantity")
*col("selling_price")).alias("Total_sales")).orderBy("order_month")


In [32]:
res4.show()

[Stage 37:==========================================================(2 + 0) / 2]

+-----------+--------------------+
|order_month|         Total_sales|
+-----------+--------------------+
|    2024-01| 4.445777757600014E8|
|    2024-02|4.1536614419999766E8|
|    2024-03| 4.436282454099968E8|
|    2024-04|4.2782097433999556E8|
|    2024-05|4.4481061894999766E8|
|    2024-06| 4.317051540600035E8|
|    2024-07| 4.436705191200028E8|
|    2024-08| 4.410951770200006E8|
|    2024-09|4.3107152608000004E8|
|    2024-10| 4.413637893100021E8|
|    2024-11|4.3362336404000014E8|
|    2024-12| 4.427129083499984E8|
+-----------+--------------------+



In [33]:
res4.write.mode("overwrite").csv("output/q4",header= True)

In [9]:
#Q5
total_order=order_items.join(products, on="product_id").groupBy("category").agg(count("quantity").alias("Total_orders"))

In [10]:
total_order.show()

[Stage 6:>                                                          (0 + 2) / 2]

+--------------+------------+
|      category|Total_orders|
+--------------+------------+
|Home & Kitchen|      434034|
|        Sports|      424412|
|   Electronics|      425896|
|      Clothing|      427607|
|         Books|      427086|
|        Beauty|      430547|
|          Toys|      430418|
+--------------+------------+



In [28]:
total_order.write.mode("overwrite").csv("temp/total",header=True)


In [29]:
total_order =  spark.read.option("header",True).csv("temp/total/t1d.csv")
total_order.createOrReplaceTempView("total_order")

In [11]:
order_category_qty= spark.sql('''
    select oi.order_id, oi.quantity, p.category
    from order_items oi join Products_table p
    on oi.product_id=p.product_id
    '''
)

In [12]:
order_category_qty.show()

+--------+--------+--------------+
|order_id|quantity|      category|
+--------+--------+--------------+
|  227444|       5|          Toys|
|   32708|       2|   Electronics|
|  426242|       5|        Beauty|
|  236965|       5|          Toys|
|  289552|       4|      Clothing|
|  631355|       4|      Clothing|
|  813168|       2|        Beauty|
|  711584|       3|        Beauty|
|  485160|       4|         Books|
|   76601|       3|Home & Kitchen|
|  531530|       2|      Clothing|
|  366262|       4|      Clothing|
|  198684|       1|         Books|
|   87222|       2|        Beauty|
|  220287|       3|Home & Kitchen|
|  567551|       5|Home & Kitchen|
|  950074|       2|      Clothing|
|  452560|       2|         Books|
|  765970|       2|      Clothing|
|  456353|       3|         Books|
+--------+--------+--------------+
only showing top 20 rows



In [13]:
order_category_qty.write.mode("overwrite").csv("temp/order_category_qty",header=True)

In [14]:
ocq =  spark.read.option("header",True).csv("temp/order_category_qty/t1b.csv")

In [15]:
ocq.createOrReplaceTempView("ocq")

In [21]:
ret_qc=spark.sql('''
select count (t.quantity) as Quantity, t.category
from returns r join ocq t
on t.order_id=r.order_id
group by t.category
''')

In [23]:
ret_qc.write.mode("overwrite").csv("temp/ret_qc",header=True)

In [25]:
ret_qc =  spark.read.option("header",True).csv("temp/ret_qc/t1c.csv")

In [26]:
ret_qc.createOrReplaceTempView("ret_qc")

In [27]:
ret_qc.show()

+--------+--------------+
|Quantity|      category|
+--------+--------------+
|   20239|Home & Kitchen|
|   19929|        Sports|
|   20169|   Electronics|
|   20035|      Clothing|
|   20059|         Books|
|   20198|        Beauty|
|   20362|          Toys|
+--------+--------------+



In [30]:
total_order.show()

+--------------+------------+
|      category|Total_orders|
+--------------+------------+
|Home & Kitchen|      434034|
|        Sports|      424412|
|   Electronics|      425896|
|      Clothing|      427607|
|         Books|      427086|
|        Beauty|      430547|
|          Toys|      430418|
+--------------+------------+



In [31]:
final=spark.sql('''
select r.category, ((r.Quantity/t.Total_orders)*100) as percentage
from ret_qc r join total_order t
on r.category = t.category
''')

In [32]:
final.show()

+--------------+------------------+
|      category|        percentage|
+--------------+------------------+
|Home & Kitchen| 4.662998751249902|
|        Sports| 4.695673072391921|
|   Electronics| 4.735663166594661|
|      Clothing| 4.685376993360726|
|         Books| 4.696712137602263|
|        Beauty|4.6912416066074085|
|          Toys| 4.730750108034516|
+--------------+------------------+



In [34]:
final.write.mode("overwrite").csv("output/q5",header=True)

In [40]:
q6=spark.sql(
    '''
    Select count(o.customer_id) as Total,c.state, o.payment_mode
    from customers c join orders o
    on o.customer_id=c.customer_id
    group by c.state,o.payment_mode
    '''
)

In [41]:
q6.show()

+-----+-----+----------------+
|Total|state|    payment_mode|
+-----+-----+----------------+
|19629|   FL|     Net Banking|
|19596|   NC|     Net Banking|
|19733|   GA|      Debit Card|
|19930|   OH|             UPI|
|20205|   MI|Cash on Delivery|
|20498|   IL|Cash on Delivery|
|20351|   OH|     Net Banking|
|19874|   TX|     Credit Card|
|20359|   IL|             UPI|
|20108|   NY|             UPI|
|20229|   NY|     Net Banking|
|20065|   TX|             UPI|
|19722|   GA|     Credit Card|
|19988|   TX|      Debit Card|
|20116|   MI|     Net Banking|
|20208|   NY|Cash on Delivery|
|20024|   CA|      Debit Card|
|20404|   IL|     Net Banking|
|19979|   CA|     Credit Card|
|19794|   OH|     Credit Card|
+-----+-----+----------------+
only showing top 20 rows



In [42]:
q6.createOrReplaceTempView("payment_counts")

In [43]:
max_count = spark.sql("""
SELECT
    state,
    MAX(Total) AS max_orders
FROM payment_counts
GROUP BY state
""")

max_count.createOrReplaceTempView("max_count")

In [44]:
max_count.show()

[Stage 57:=============================>                            (1 + 1) / 2]

+-----+----------+
|state|max_orders|
+-----+----------+
|   MI|     20416|
|   CA|     20246|
|   NC|     19596|
|   IL|     20498|
|   WA|     20244|
|   OH|     20351|
|   NY|     20369|
|   TX|     20065|
|   GA|     20041|
|   FL|     20010|
+-----+----------+



In [47]:
finale= spark.sql('''
SELECT
    p.state,
    p.payment_mode,
    p.Total
FROM payment_counts p
JOIN max_count m
ON p.state = m.state
AND p.Total = m.max_orders
''')

In [48]:
finale.show()

[Stage 73:=============================>                            (1 + 1) / 2]

+-----+----------------+-----+
|state|    payment_mode|Total|
+-----+----------------+-----+
|   NC|     Net Banking|19596|
|   IL|Cash on Delivery|20498|
|   OH|     Net Banking|20351|
|   TX|             UPI|20065|
|   GA|     Net Banking|20041|
|   WA|             UPI|20244|
|   CA|             UPI|20246|
|   FL|      Debit Card|20010|
|   NY|      Debit Card|20369|
|   MI|      Debit Card|20416|
+-----+----------------+-----+



In [49]:
finale.write.mode("overwrite").csv("output/q6",header=True)